# 🔍 Exploratory Data Analysis — DSC MCF ITB 2026
## Tim: Clair Obscur
### Asuransi Kesehatan Individu | AXA Financial Indonesia

Notebook ini berisi seluruh proses EDA sesuai **Case Document** dan template **Laporan** (Bab III Subbab 3.1).

Struktur EDA:
1. [Load Data & Setup](#1)
2. [Overview Dataset](#2)
3. [Reporting Lag: Masuk RS vs Tanggal Pembayaran](#3)
4. [Tren Frekuensi & Severitas Bulanan (Gambar 3.1)](#4)
5. [Faktor Penentu Severitas & Distribusi Lokasi RS (Gambar 3.2)](#5)
6. [Kontribusi Nilai vs Volume per Kelompok Penyakit ICD (Gambar 3.3)](#6)
7. [Analisis Outlier (Klaim > Rp 1 Miliar)](#7)
8. [Profil Demografis: Usia, Gender, Plan Code](#8)
9. [Distribusi Frekuensi Klaim per Polis](#9)
10. [Heatmap Korelasi & Length of Stay](#10)

## 1. Load Data & Setup <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 9,  'ytick.labelsize': 9,
    'figure.dpi': 120,
    'axes.spines.top': False, 'axes.spines.right': False,
})
PALETTE   = ['#2196F3','#FF5722','#4CAF50','#9C27B0','#FF9800','#00BCD4','#795548','#607D8B']
COLOR_FREQ  = '#1565C0'
COLOR_SEV   = '#C62828'
COLOR_TOTAL = '#2E7D32'
COLOR_MA    = '#FF6F00'
GRID_KW     = dict(color='#E0E0E0', lw=0.5)

# ── Load
# Sesuaikan path jika dijalankan di Kaggle atau lokal
KLAIM_PATH = '/kaggle/input/datasets/yosiajosephchandra/mcfitb-clairobscur/Data_Klaim.csv'
POLIS_PATH = '/kaggle/input/datasets/yosiajosephchandra/mcfitb-clairobscur/Data_Polis.csv'

klaim = pd.read_csv(KLAIM_PATH)
polis = pd.read_csv(POLIS_PATH)

# Parse tanggal
klaim['Tanggal Pasien Masuk RS']  = pd.to_datetime(klaim['Tanggal Pasien Masuk RS'])
klaim['Tanggal Pasien Keluar RS'] = pd.to_datetime(klaim['Tanggal Pasien Keluar RS'])
klaim['Tanggal Pembayaran Klaim'] = pd.to_datetime(klaim['Tanggal Pembayaran Klaim'])
polis['BirthDate'] = pd.to_datetime(polis['Tanggal Lahir'].astype(str), format='%Y%m%d', errors='coerce')
polis['EffDate']   = pd.to_datetime(polis['Tanggal Efektif Polis'].astype(str), format='%Y%m%d', errors='coerce')

# Derived columns
klaim['LOS']      = (klaim['Tanggal Pasien Keluar RS'] - klaim['Tanggal Pasien Masuk RS']).dt.days
klaim['Delay']    = (klaim['Tanggal Pembayaran Klaim']  - klaim['Tanggal Pasien Masuk RS']).dt.days
klaim['YearMonth']= klaim['Tanggal Pasien Masuk RS'].dt.to_period('M')

# ICD grouping
ICD_MAP = {'C':'Kanker','N':'Urogenital/Ginjal','I':'Kardiovaskular',
           'K':'Pencernaan','H':'Mata & THT','M':'Muskuloskeletal',
           'J':'Pernapasan','A':'Infeksi','B':'Infeksi'}
klaim['ICD_Group'] = klaim['ICD Diagnosis'].str[0].map(ICD_MAP).fillna('Lainnya')

# Merge demografis
merged = klaim.merge(
    polis[['Nomor Polis','BirthDate','Gender','Plan Code','Domisili']],
    on='Nomor Polis', how='left')
merged['Age'] = ((merged['Tanggal Pasien Masuk RS'] - merged['BirthDate']).dt.days / 365.25)

print(f"✅ Data Klaim  : {klaim.shape[0]:,} baris × {klaim.shape[1]} kolom")
print(f"✅ Data Polis  : {polis.shape[0]:,} baris × {polis.shape[1]} kolom")
print(f"✅ Data Merged : {merged.shape[0]:,} baris")

## 2. Overview Dataset <a id='2'></a>

In [ ]:
print("=" * 60)
print("RINGKASAN DATA KLAIM")
print("=" * 60)
print(f"  Total klaim           : {len(klaim):,}")
print(f"  Rentang masuk RS      : {klaim['Tanggal Pasien Masuk RS'].min().date()} s/d {klaim['Tanggal Pasien Masuk RS'].max().date()}")
print(f"  Polis unik klaim      : {klaim['Nomor Polis'].nunique():,} dari {len(polis):,} polis ({klaim['Nomor Polis'].nunique()/len(polis)*100:.1f}%)")
print(f"  Claim rate            : {klaim['Nomor Polis'].nunique()/len(polis)*100:.1f}% polis melakukan klaim")
print(f"  Total nilai klaim     : Rp {klaim['Nominal Klaim Yang Disetujui'].sum()/1e9:.2f} miliar")
print(f"  Rata-rata severitas   : Rp {klaim['Nominal Klaim Yang Disetujui'].mean()/1e6:.2f} juta / klaim")
print(f"  Median severitas      : Rp {klaim['Nominal Klaim Yang Disetujui'].median()/1e6:.2f} juta / klaim")
print()
print("MISSING VALUES:")
print(klaim.isnull().sum()[klaim.isnull().sum() > 0])
print()
print("DISTRIBUSI LOKASI RS:")
print(klaim['Lokasi RS'].value_counts())
print()
print("DISTRIBUSI TIPE PERAWATAN:")
print(klaim['Inpatient/Outpatient'].value_counts())
print()
print("DISTRIBUSI METODE PEMBAYARAN:")
print(klaim['Reimburse/Cashless'].value_counts())

In [ ]:
# Tampilkan 5 baris pertama
display(klaim.head())
display(polis.head())

## 3. Reporting Lag: Masuk RS vs Tanggal Pembayaran <a id='3'></a>

> **Temuan kunci:** Rata-rata jeda antara tanggal masuk RS dan tanggal pembayaran klaim adalah **±66 hari** (median 62 hari).
> Oleh karena itu, agregasi bulanan **wajib** menggunakan *Tanggal Pasien Masuk RS*, bukan Tanggal Pembayaran, untuk menghindari distorsi *reporting lag*.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle('Analisis Delay Pembayaran Klaim (Tanggal Pembayaran − Tanggal Masuk RS)',
             fontsize=12, fontweight='bold')

delay_clean = klaim['Delay'].dropna().clip(0, 365)
ax = axes[0]
ax.hist(delay_clean, bins=40, color=COLOR_FREQ, alpha=0.75, edgecolor='white')
ax.axvline(klaim['Delay'].median(), color=COLOR_SEV, lw=2, linestyle='--',
           label=f"Median: {klaim['Delay'].median():.0f} hari")
ax.axvline(klaim['Delay'].mean(), color='orange', lw=2, linestyle='-.',
           label=f"Mean: {klaim['Delay'].mean():.0f} hari")
ax.set_xlabel('Delay (hari)'); ax.set_ylabel('Jumlah Klaim')
ax.set_title('(a) Distribusi Delay Pembayaran')
ax.legend(fontsize=9); ax.grid(**GRID_KW)

# Monthly masuk RS vs Pembayaran
klaim_pay = klaim.dropna(subset=['Tanggal Pembayaran Klaim']).copy()
klaim_pay['YM_Pay'] = klaim_pay['Tanggal Pembayaran Klaim'].dt.to_period('M')
monthly_masuk = klaim.groupby('YearMonth').size().reset_index(name='count_masuk')
monthly_pay   = klaim_pay.groupby('YM_Pay').size().reset_index(name='count_pay')
monthly_masuk['MonthStr'] = monthly_masuk['YearMonth'].astype(str)
monthly_pay['MonthStr']   = monthly_pay['YM_Pay'].astype(str)

ax = axes[1]
all_months = sorted(set(monthly_masuk['MonthStr']) | set(monthly_pay['MonthStr']))
m_dict = {m: i for i, m in enumerate(all_months)}
ax.plot([m_dict[m] for m in monthly_masuk['MonthStr']], monthly_masuk['count_masuk'],
        marker='o', color=COLOR_FREQ, label='Berdasarkan Masuk RS', lw=2)
ax.plot([m_dict[m] for m in monthly_pay['MonthStr']], monthly_pay['count_pay'],
        marker='s', color=COLOR_SEV, label='Berdasarkan Pembayaran', lw=2, linestyle='--')
tick_pos2 = list(range(0, len(all_months), 3))
ax.set_xticks(tick_pos2)
ax.set_xticklabels([all_months[i][-5:] for i in tick_pos2], rotation=45, ha='right')
ax.set_title('(b) Volume Klaim per Bulan: Masuk RS vs Pembayaran')
ax.set_ylabel('Jumlah Klaim'); ax.legend(fontsize=9); ax.grid(**GRID_KW)
ax.text(0.5, 0.05,
        '→ Reporting lag rata-rata 62 hari.\nGunakan Masuk RS untuk agregasi.',
        transform=ax.transAxes, ha='center', fontsize=8.5, color='#555',
        bbox=dict(facecolor='#FFF9C4', alpha=0.8, boxstyle='round'))

plt.tight_layout(); plt.show()
print(f"Delay stats: mean={klaim['Delay'].mean():.1f} hari, median={klaim['Delay'].median():.0f} hari")

## 4. Tren Frekuensi & Severitas Bulanan (Gambar 3.1) <a id='4'></a>

> **Agregasi** menggunakan tanggal masuk RS. Klaim > Rp 1 miliar di-*cap* untuk menghindari distorsi pada deret waktu.

In [ ]:
# Agregasi bulanan (dengan cap outlier > 1 miliar)
klaim_cap = klaim.copy()
klaim_cap['Nominal Klaim Yang Disetujui'] = klaim_cap['Nominal Klaim Yang Disetujui'].clip(upper=1e9)

monthly = klaim_cap.groupby('YearMonth').agg(
    Frequency=('Claim ID','count'),
    Total=('Nominal Klaim Yang Disetujui','sum')
).reset_index()
monthly['Severity']   = monthly['Total'] / monthly['Frequency']
monthly['MonthStr']   = monthly['YearMonth'].astype(str)
monthly['MA3_Freq']   = monthly['Frequency'].rolling(3, min_periods=1).mean()
monthly['MA3_Sev']    = monthly['Severity'].rolling(3,  min_periods=1).mean()

display(monthly[['MonthStr','Frequency','Severity','Total']].style.format({
    'Frequency': '{:.0f}',
    'Severity' : 'Rp {:,.0f}',
    'Total'    : 'Rp {:,.0f}',
}))

In [ ]:
x      = range(len(monthly))
labels = [m[-5:] for m in monthly['MonthStr']]
tick_pos = list(range(0, len(monthly), 2))
tick_lab = [labels[i] for i in tick_pos]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle(
    'Tren Frekuensi dan Severitas Klaim Bulanan | Jan 2024 – Jul 2025\n'
    '(berdasarkan tanggal masuk RS; outlier > Rp 1 miliar dikeluarkan)',
    fontsize=12, fontweight='bold', y=1.01)

# ── (a) Frekuensi
ax = axes[0]
ax.plot(x, monthly['Frequency'], marker='o', color=COLOR_FREQ, lw=2, label='Frekuensi', zorder=3)
ax.plot(x, monthly['MA3_Freq'], linestyle='--', color=COLOR_MA, lw=1.5, label='MA 3 Bulan')
ax.axhline(monthly['Frequency'].mean(), linestyle=':', color='gray', lw=1,
           label=f"Rata-rata {monthly['Frequency'].mean():.0f}")
ax.fill_between(x, monthly['Frequency'].mean(), monthly['Frequency'],
                where=monthly['Frequency'] >= monthly['Frequency'].mean(),
                alpha=0.08, color=COLOR_FREQ)

# Annotasi
for i, row in monthly.iterrows():
    if row['MonthStr'] in ['2024-01','2025-07','2024-11','2025-01']:
        val = int(row['Frequency'])
        offset = 12 if val > monthly['Frequency'].mean() else -20
        ax.annotate(f"{val} klaim", xy=(i, val), xytext=(i, val+offset),
                    fontsize=7.5, ha='center', color=COLOR_FREQ,
                    arrowprops=dict(arrowstyle='->', color=COLOR_FREQ, lw=0.8))

# H1 bands
ax.axvspan(-0.5, 5.5, alpha=0.04, color='blue')
ax.text(2.5, monthly['Frequency'].max()*1.03, 'H1 2024\n252/bln', ha='center', fontsize=8, color='steelblue')
ax.axvspan(12.5, 17.5, alpha=0.04, color='orange')
ax.text(15, monthly['Frequency'].max()*1.03, 'H1 2025\n229/bln', ha='center', fontsize=8, color='darkorange')

ax.set_title('(a) Frekuensi Klaim Bulanan', fontweight='bold')
ax.set_ylabel('Frekuensi Klaim (klaim/bulan)')
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_lab, rotation=45, ha='right')
ax.grid(**GRID_KW); ax.legend(fontsize=8, framealpha=0.7); ax.set_ylim(150, 365)

# ── (b) Severitas
ax = axes[1]
ax.plot(x, monthly['Severity']/1e6, marker='s', color=COLOR_SEV, lw=2, label='Severitas', zorder=3)
ax.plot(x, monthly['MA3_Sev']/1e6, linestyle='--', color=COLOR_MA, lw=1.5, label='MA 3 Bulan')
avg_sev = monthly['Severity'].mean()/1e6
ax.axhline(avg_sev, linestyle=':', color='gray', lw=1, label=f'Rata-rata Rp {avg_sev:.1f}M')
ax.fill_between(x, avg_sev, monthly['Severity']/1e6,
                where=monthly['Severity']/1e6 >= avg_sev, alpha=0.08, color=COLOR_SEV)
ax.fill_between(x, avg_sev, monthly['Severity']/1e6,
                where=monthly['Severity']/1e6 < avg_sev, alpha=0.08, color='steelblue')

for i, row in monthly.iterrows():
    if row['MonthStr'] in ['2024-01','2025-02','2024-12']:
        val = row['Severity']/1e6
        offset = 2.5 if val > avg_sev else -4
        ax.annotate(f"Rp {val:.1f}M", xy=(i, val), xytext=(i, val+offset),
                    fontsize=7.5, ha='center', color=COLOR_SEV,
                    arrowprops=dict(arrowstyle='->', color=COLOR_SEV, lw=0.8))

ax.set_title('(b) Severitas Klaim Bulanan', fontweight='bold')
ax.set_ylabel('Severitas (Rp juta/klaim)')
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_lab, rotation=45, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp {x:.0f}M'))
ax.grid(**GRID_KW); ax.legend(fontsize=8, framealpha=0.7)

plt.tight_layout(); plt.show()

# Key stats
print(f"Rata-rata frekuensi  : {monthly['Frequency'].mean():.1f} klaim/bulan")
print(f"Maks frekuensi       : {monthly['Frequency'].max():.0f} ({monthly.loc[monthly['Frequency'].idxmax(),'MonthStr']})")
print(f"Min  frekuensi       : {monthly['Frequency'].min():.0f} ({monthly.loc[monthly['Frequency'].idxmin(),'MonthStr']})")
print(f"Rata-rata severitas  : Rp {monthly['Severity'].mean()/1e6:.2f} juta")

## 5. Faktor Penentu Severitas & Distribusi Lokasi RS (Gambar 3.2) <a id='5'></a>

Tiga faktor dominan yang menjelaskan **88% variasi severitas**:
1. **Lokasi RS (Singapore)** — 42.26%
2. **Length of Stay (LOS)** — 35.37%
3. **Usia Pasien** — 10.78%

In [ ]:
# ── Feature importance (dari hasil RF analysis)
features    = ['Lokasi RS (Singapore)', 'Lama Rawat Inap / LOS', 'Usia Pasien',
               'Tipe Perawatan (IP/OP/ODC)', 'Kategori Diagnosis (ICD)',
               'Metode Pembayaran', 'Bulan Klaim', 'Plan Code',
               'Domisili Pemegang Polis', 'Jenis Kelamin']
importances = [42.26, 35.37, 10.78, 4.92, 2.81, 1.43, 0.93, 0.72, 0.42, 0.34]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle('Faktor Penentu Nilai Klaim (Severitas)', fontsize=13, fontweight='bold')

# (a) Bar feature importance
ax = axes[0]
colors_bar = ['#C62828' if i < 3 else '#90A4AE' for i in range(len(features))]
bars = ax.barh(features[::-1], importances[::-1], color=colors_bar[::-1], edgecolor='white', height=0.7)
for bar, val in zip(bars, importances[::-1]):
    ax.text(bar.get_width()+0.4, bar.get_y()+bar.get_height()/2,
            f'{val:.2f}%', va='center', ha='left',
            fontsize=9, fontweight='bold' if val >= 10 else 'normal')
ax.set_xlabel('Feature Importance (%)'); ax.set_xlim(0, 50)
ax.set_title('(a) Feature Importance — Faktor Penentu\nSeveritas Klaim', fontweight='bold')
ax.grid(axis='x', **GRID_KW)
ax.annotate('Top 3 fitur\n= 88.4% variasi\nseveritas klaim',
            xy=(importances[0], len(features)-1), xytext=(32, len(features)-4),
            fontsize=8, color='#C62828', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#C62828'))

# (b) Box plot lokasi RS
ax = axes[1]
main_locs = klaim[klaim['Lokasi RS'].isin(['Indonesia','Malaysia','Singapore'])].copy()
loc_order = ['Indonesia','Malaysia','Singapore']
pal_loc   = {'Indonesia':'#2196F3', 'Malaysia':'#FF9800', 'Singapore':'#C62828'}

for i, loc in enumerate(loc_order):
    data = main_locs.loc[main_locs['Lokasi RS']==loc, 'Nominal Klaim Yang Disetujui'].dropna()
    data = data[data > 0]
    ax.boxplot(data, positions=[i], widths=0.5, patch_artist=True,
               medianprops=dict(color='white', lw=2),
               boxprops=dict(facecolor=pal_loc[loc], alpha=0.7),
               whiskerprops=dict(color=pal_loc[loc]),
               capprops=dict(color=pal_loc[loc]),
               flierprops=dict(marker='.', color=pal_loc[loc], alpha=0.3, ms=3))
    med = data.median()
    ax.text(i, med*1.4, f"Median\nRp {med/1e6:.1f}M", ha='center', fontsize=8.5,
            color='white', fontweight='bold',
            bbox=dict(facecolor=pal_loc[loc], alpha=0.8, boxstyle='round,pad=0.2'))

ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'Rp {x/1e9:.1f}M' if x>=1e9 else (f'Rp {x/1e6:.0f}M' if x>=1e6 else f'Rp {x/1e3:.0f}K')))
ax.set_xticks([0,1,2])
n_per = {l: len(main_locs[main_locs['Lokasi RS']==l]) for l in loc_order}
ax.set_xticklabels([f'{l}\n(n={n_per[l]:,})' for l in loc_order])
ax.set_title('(b) Distribusi Nilai Klaim per\nLokasi Rumah Sakit', fontweight='bold')
ax.set_ylabel('Nominal Klaim Disetujui (Rp, skala log)')
ax.grid(axis='y', **GRID_KW)
ax.annotate('Singapore\n6x lebih tinggi\ndari Indonesia',
            xy=(2, main_locs.loc[main_locs['Lokasi RS']=='Singapore','Nominal Klaim Yang Disetujui'].median()),
            xytext=(1.3, 2e8), fontsize=8, color='#C62828', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#C62828'))

plt.tight_layout(); plt.show()

sg_med = main_locs.loc[main_locs['Lokasi RS']=='Singapore','Nominal Klaim Yang Disetujui'].median()
id_med = main_locs.loc[main_locs['Lokasi RS']=='Indonesia','Nominal Klaim Yang Disetujui'].median()
sg_pct_freq = len(main_locs[main_locs['Lokasi RS']=='Singapore'])/len(klaim)*100
sg_pct_val  = main_locs.loc[main_locs['Lokasi RS']=='Singapore','Nominal Klaim Yang Disetujui'].sum()/klaim['Nominal Klaim Yang Disetujui'].sum()*100
print(f"Median SG  : Rp {sg_med/1e6:.1f} juta")
print(f"Median ID  : Rp {id_med/1e6:.1f} juta")
print(f"Ratio SG/ID: {sg_med/id_med:.1f}x")
print(f"SG % freq  : {sg_pct_freq:.1f}%")
print(f"SG % nilai : {sg_pct_val:.1f}%")

## 6. Kontribusi Nilai vs Volume per Kelompok Penyakit ICD (Gambar 3.3) <a id='6'></a>

> Kanker dan Kardiovaskular secara bersama-sama menyumbang **46% total nilai klaim** meski hanya **31.5% volume**.

In [ ]:
grp = klaim.groupby('ICD_Group').agg(
    Volume=('Claim ID','count'),
    TotalNilai=('Nominal Klaim Yang Disetujui','sum')
).reset_index()
grp['Pct_Vol']   = grp['Volume']   / grp['Volume'].sum()   * 100
grp['Pct_Nilai'] = grp['TotalNilai']/ grp['TotalNilai'].sum()* 100
grp['Leverage']  = grp['Pct_Nilai'] - grp['Pct_Vol']
grp = grp.sort_values('TotalNilai', ascending=False).reset_index(drop=True)

display(grp.style.format({
    'Volume':'    {:,}',
    'TotalNilai':'Rp {:,.0f}',
    'Pct_Vol':'{:.1f}%',
    'Pct_Nilai':'{:.1f}%',
    'Leverage':'{:+.1f}pp'
}))

In [ ]:
colors_icd = ['#C62828','#FF7043','#1565C0','#2E7D32','#F57F17','#6A1B9A','#00838F','#4E342E','#37474F']

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
fig.suptitle(
    'Kontribusi Nilai vs Volume Klaim per Kelompok Penyakit (ICD)\n'
    'Jan 2024 – Jul 2025  |  n = 4,627 klaim  |  outlier > Rp 1 miliar dikeluarkan',
    fontsize=11, fontweight='bold')

# (a) Pie Nilai
ax = axes[0]
wedges, _, autotexts = ax.pie(grp['Pct_Nilai'], labels=None, autopct='%1.1f%%',
    colors=colors_icd[:len(grp)], startangle=90,
    wedgeprops=dict(edgecolor='white', lw=1.2), pctdistance=0.72)
for at in autotexts: at.set_fontsize(8)
ax.set_title('(a) Nilai Klaim\n(% dari Rp 254.6 M)', fontweight='bold')
kk_nilai = grp.loc[grp['ICD_Group'].isin(['Kanker','Kardiovaskular']),'Pct_Nilai'].sum()
ax.text(0, -1.4, f'Kanker + Kardiovaskular\n= {kk_nilai:.1f}% dari total nilai',
        ha='center', fontsize=9, color='#C62828', fontweight='bold')

# (b) Pie Volume
ax = axes[1]
_, _, autotexts2 = ax.pie(grp['Pct_Vol'], labels=None, autopct='%1.1f%%',
    colors=colors_icd[:len(grp)], startangle=90,
    wedgeprops=dict(edgecolor='white', lw=1.2), pctdistance=0.72)
for at in autotexts2: at.set_fontsize(8)
ax.set_title('(b) Volume Klaim\n(% dari 4,627 klaim)', fontweight='bold')
kk_vol = grp.loc[grp['ICD_Group'].isin(['Kanker','Kardiovaskular']),'Pct_Vol'].sum()
ax.text(0, -1.4, f'Kanker + Kardiovaskular\n= {kk_vol:.1f}% dari total volume',
        ha='center', fontsize=9, color='#555', fontweight='bold')

fig.legend(wedges, grp['ICD_Group'], loc='lower center', ncol=5, fontsize=8,
           bbox_to_anchor=(0.5, -0.06), framealpha=0.7)

# (c) Leverage bar
ax = axes[2]
lev = grp.sort_values('Leverage')
bar_cols = ['#C62828' if v > 0 else '#1565C0' for v in lev['Leverage']]
ax.barh(lev['ICD_Group'], lev['Leverage'], color=bar_cols, edgecolor='white')
ax.axvline(0, color='black', lw=1)
for i, (_, row) in enumerate(lev.iterrows()):
    ax.text(row['Leverage'] + (0.2 if row['Leverage'] >= 0 else -0.2), i,
            f"+{row['Leverage']:.1f}pp" if row['Leverage'] > 0 else f"{row['Leverage']:.1f}pp",
            va='center', ha='left' if row['Leverage'] >= 0 else 'right', fontsize=8.5)
ax.set_title('(c) Leverage\n% Nilai − % Volume', fontweight='bold')
ax.set_xlabel('Leverage (percentage point)'); ax.grid(axis='x', **GRID_KW)

plt.tight_layout(); plt.show()

## 7. Analisis Outlier (Klaim > Rp 1 Miliar) <a id='7'></a>

> **17 klaim** (0.4% volume) menyumbang **9.5% total nilai** → perlu mekanisme *stop-loss* dalam pengelolaan risiko portofolio.

In [ ]:
outlier = klaim[klaim['Nominal Klaim Yang Disetujui'] > 1e9].copy()
normal  = klaim[klaim['Nominal Klaim Yang Disetujui'] <= 1e9].copy()

print(f"Klaim outlier (> Rp 1 M) : {len(outlier):,} klaim ({len(outlier)/len(klaim)*100:.2f}%)")
print(f"Total nilai outlier      : Rp {outlier['Nominal Klaim Yang Disetujui'].sum()/1e9:.3f} miliar")
print(f"Kontribusi thd total     : {outlier['Nominal Klaim Yang Disetujui'].sum()/klaim['Nominal Klaim Yang Disetujui'].sum()*100:.1f}%")
print()
print("Detail klaim outlier:")
display(outlier[['Claim ID','Nomor Polis','ICD Diagnosis','ICD Description',
                 'Lokasi RS','Nominal Klaim Yang Disetujui','LOS']].sort_values(
    'Nominal Klaim Yang Disetujui', ascending=False).style.format({'Nominal Klaim Yang Disetujui':'Rp {:,.0f}'}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle('Analisis Klaim Outlier (> Rp 1 Miliar)', fontsize=12, fontweight='bold')

# (a) Impact bar
ax = axes[0]
cats = ['Normal\n(≤ Rp 1 M)', 'Outlier\n(> Rp 1 M)']
n_total = len(klaim)
pcts_freq = [(len(normal)/n_total*100), (len(outlier)/n_total*100)]
pcts_val  = [normal['Nominal Klaim Yang Disetujui'].sum()/klaim['Nominal Klaim Yang Disetujui'].sum()*100,
             outlier['Nominal Klaim Yang Disetujui'].sum()/klaim['Nominal Klaim Yang Disetujui'].sum()*100]
x_bar = [0, 1]; width = 0.35
ax.bar([x-width/2 for x in x_bar], pcts_freq, width, label='% Frekuensi',
       color=['#2196F3','#C62828'], alpha=0.8, edgecolor='white')
ax.bar([x+width/2 for x in x_bar], pcts_val,  width, label='% Total Nilai',
       color=['#4CAF50','#FF5722'], alpha=0.8, edgecolor='white')
ax.set_xticks(x_bar); ax.set_xticklabels(cats)
ax.set_ylabel('%'); ax.set_title('(a) Dampak Klaim Outlier\n(% Frekuensi vs % Total Nilai)')
ax.legend(fontsize=9); ax.grid(axis='y', **GRID_KW)
for xi, vals in zip(x_bar, zip(pcts_freq, pcts_val)):
    for j, v in enumerate(vals):
        ax.text(xi+(j-0.5)*width, v+0.5, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

# (b) Outlier by ICD group
ax = axes[1]
out_icd = outlier['ICD_Group'].value_counts()
ax.bar(out_icd.index, out_icd.values, color='#C62828', alpha=0.8, edgecolor='white')
ax.set_title(f'(b) Distribusi ICD Klaim Outlier\n(n={len(outlier)} klaim)')
ax.set_xlabel('Kelompok ICD'); ax.set_ylabel('Jumlah Klaim')
ax.grid(axis='y', **GRID_KW)
for i, v in enumerate(out_icd.values):
    ax.text(i, v+0.05, str(v), ha='center', fontsize=10, fontweight='bold')

# (c) Outlier by location
ax = axes[2]
out_loc = outlier['Lokasi RS'].value_counts()
colors_loc = ['#C62828' if l=='Singapore' else '#2196F3' if l=='Indonesia' else '#FF9800'
              for l in out_loc.index]
ax.bar(out_loc.index, out_loc.values, color=colors_loc, alpha=0.8, edgecolor='white')
ax.set_title(f'(c) Distribusi Lokasi RS Klaim Outlier\n(n={len(outlier)} klaim)')
ax.set_xlabel('Lokasi RS'); ax.set_ylabel('Jumlah Klaim')
ax.grid(axis='y', **GRID_KW)
for i, v in enumerate(out_loc.values):
    ax.text(i, v+0.05, str(v), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout(); plt.show()

## 8. Profil Demografis: Usia, Gender, Plan Code <a id='8'></a>

In [ ]:
merged_clean = merged.dropna(subset=['Age'])
merged_clean = merged_clean[(merged_clean['Age'] > 0) & (merged_clean['Age'] < 100)]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle('Profil Demografis Nasabah Klaim', fontsize=12, fontweight='bold')

# (a) Usia
ax = axes[0]
age_bins   = [0, 20, 30, 40, 50, 60, 70, 100]
age_labels = ['<20','20-30','30-40','40-50','50-60','60-70','>70']
counts, _  = np.histogram(merged_clean['Age'], bins=age_bins)
bars = ax.bar(age_labels, counts, color=PALETTE[:7], edgecolor='white')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
            str(count), ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Kelompok Usia'); ax.set_ylabel('Jumlah Klaim')
ax.set_title(f'(a) Distribusi Usia Pasien\n(rata-rata {merged_clean["Age"].mean():.1f} tahun)')
ax.grid(axis='y', **GRID_KW)

# (b) Gender
ax = axes[1]
gender_klaim = merged['Gender'].value_counts()
gender_polis = polis['Gender'].value_counts()
x_g = [0, 1]; wid = 0.35
ax.bar([x-wid/2 for x in x_g], [gender_polis.get('M',0)/len(polis)*100, gender_polis.get('F',0)/len(polis)*100],
       wid, label='Polis', color=['#1565C0','#E91E63'], alpha=0.7, edgecolor='white')
ax.bar([x+wid/2 for x in x_g], [gender_klaim.get('M',0)/len(merged)*100, gender_klaim.get('F',0)/len(merged)*100],
       wid, label='Klaim', color=['#42A5F5','#F48FB1'], alpha=0.9, edgecolor='white')
ax.set_xticks(x_g); ax.set_xticklabels(['Laki-laki (M)','Perempuan (F)'])
ax.set_ylabel('%'); ax.set_title('(b) Distribusi Gender\nPolis vs Klaim')
ax.legend(fontsize=9); ax.grid(axis='y', **GRID_KW)

# (c) Plan Code
ax = axes[2]
plan_klaim = merged['Plan Code'].value_counts()
plan_polis = polis['Plan Code'].value_counts()
plans = ['M-001','M-002','M-003']
x_p   = range(len(plans)); wid = 0.35
ax.bar([x-wid/2 for x in x_p], [plan_polis.get(p,0)/len(polis)*100 for p in plans],
       wid, label='Polis', color='#1565C0', alpha=0.7, edgecolor='white')
ax.bar([x+wid/2 for x in x_p], [plan_klaim.get(p,0)/len(merged)*100 for p in plans],
       wid, label='Klaim', color='#42A5F5', alpha=0.9, edgecolor='white')
ax.set_xticks(x_p); ax.set_xticklabels(plans)
ax.set_ylabel('%'); ax.set_title('(c) Distribusi Plan Code\nPolis vs Klaim')
ax.legend(fontsize=9); ax.grid(axis='y', **GRID_KW)

plt.tight_layout(); plt.show()

print("\nDistribusi Plan Code (Polis):")
print(polis['Plan Code'].value_counts())
print("\nDistribusi Plan Code (Klaim):")
print(merged['Plan Code'].value_counts())

## 9. Distribusi Frekuensi Klaim per Polis <a id='9'></a>

In [ ]:
freq_per_polis = klaim.groupby('Nomor Polis').size().reset_index(name='n_klaim')
n_polis_klaim  = len(freq_per_polis)

print(f"Polis dengan ≥1 klaim : {n_polis_klaim:,} dari {len(polis):,} ({n_polis_klaim/len(polis)*100:.1f}%)")
print(f"Polis tanpa klaim     : {len(polis)-n_polis_klaim:,} ({(len(polis)-n_polis_klaim)/len(polis)*100:.1f}%)")
print(f"\nStatistik klaim per polis:")
display(freq_per_polis['n_klaim'].describe().to_frame().T.style.format('{:.2f}'))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle('Distribusi Frekuensi Klaim per Polis', fontsize=12, fontweight='bold')

ax = axes[0]
claim_bins   = [0,1,2,3,5,10,20,250]
claim_labels = ['1','2','3','4-5','6-10','11-20','>20']
counts2, _ = np.histogram(freq_per_polis['n_klaim'], bins=claim_bins)
bars2 = ax.bar(claim_labels, counts2, color=PALETTE[:7], edgecolor='white')
for bar, count in zip(bars2, counts2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            str(count), ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Jumlah Klaim per Polis'); ax.set_ylabel('Jumlah Polis')
ax.set_title(f'(a) Distribusi Jumlah Klaim per Polis\n'
             f'median={freq_per_polis["n_klaim"].median():.0f} | maks={freq_per_polis["n_klaim"].max():.0f}')
ax.grid(axis='y', **GRID_KW)

ax = axes[1]
# Pareto: top 10% polis, berapa % klaim?
freq_sorted = freq_per_polis.sort_values('n_klaim', ascending=False).reset_index(drop=True)
freq_sorted['pct_polis'] = (freq_sorted.index+1)/len(freq_sorted)*100
freq_sorted['cum_klaim'] = freq_sorted['n_klaim'].cumsum()/freq_sorted['n_klaim'].sum()*100
ax.plot(freq_sorted['pct_polis'], freq_sorted['cum_klaim'], color=COLOR_FREQ, lw=2)
ax.axvline(10, color='gray', linestyle='--', lw=1)
ax.axhline(freq_sorted.loc[freq_sorted['pct_polis']<=10,'cum_klaim'].max(),
           color='gray', linestyle='--', lw=1)
top10_pct = freq_sorted.loc[freq_sorted['pct_polis']<=10,'cum_klaim'].max()
ax.text(12, top10_pct-5, f'Top 10% polis\n→ {top10_pct:.1f}% klaim', fontsize=9, color=COLOR_FREQ)
ax.set_xlabel('% Polis (dari tertinggi ke terendah)')
ax.set_ylabel('% Kumulatif Klaim')
ax.set_title('(b) Kurva Pareto: Konsentrasi Klaim per Polis')
ax.grid(**GRID_KW)

plt.tight_layout(); plt.show()

## 10. Heatmap Korelasi & Length of Stay <a id='10'></a>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Analisis Korelasi: Lokasi RS × ICD Group dan LOS vs Severitas',
             fontsize=12, fontweight='bold')

pivot = klaim[klaim['Lokasi RS'].isin(['Indonesia','Malaysia','Singapore'])].copy()
heat_data = pivot.groupby(['Lokasi RS','ICD_Group'])['Nominal Klaim Yang Disetujui'].median().unstack(fill_value=0)
ax = axes[0]
sns.heatmap(heat_data/1e6, annot=True, fmt='.0f', cmap='YlOrRd',
            ax=ax, linewidths=0.5, cbar_kws={'label':'Median Klaim (Rp Juta)'})
ax.set_title('(a) Median Nilai Klaim (Rp Juta)\nper Lokasi RS × Kelompok Diagnosis')
ax.set_xlabel('Kelompok Diagnosis (ICD)'); ax.set_ylabel('Lokasi RS')

ax = axes[1]
for loc, color in [('Indonesia','#2196F3'),('Malaysia','#FF9800'),('Singapore','#C62828')]:
    sub = klaim[(klaim['Lokasi RS']==loc) & (klaim['LOS']>=0) & (klaim['LOS']<=30)]
    ax.scatter(sub['LOS'], sub['Nominal Klaim Yang Disetujui']/1e6,
               alpha=0.25, color=color, label=loc, s=20)
ax.set_xlabel('Length of Stay (hari)')
ax.set_ylabel('Nominal Klaim (Rp Juta)')
ax.set_title('(b) Length of Stay vs Nilai Klaim\nper Lokasi RS')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rp {x:.0f}M'))
ax.legend(fontsize=9); ax.grid(**GRID_KW)

plt.tight_layout(); plt.show()

# LOS stats
print("LOS (Length of Stay) Statistics:")
display(klaim.groupby('Lokasi RS')['LOS'].describe().loc[['Indonesia','Malaysia','Singapore']].style.format('{:.1f}'))

## 📋 Ringkasan Temuan EDA

| Aspek | Temuan Utama |
|---|---|
| **Klaim Rate** | Hanya 29.5% polis (1.210 dari 4.096) yang pernah klaim |
| **Frekuensi Bulanan** | Rata-rata 244 klaim/bulan; puncak Jan 2024 (302) dan Jul 2025 (264) |
| **Pola Musiman** | Frekuensi memuncak di Januari dan Oktober-November |
| **Tren Tahunan** | H1 2024 rata-rata 252/bln → turun ke 229/bln di H1 2025 (−9.4%) |
| **Severitas** | Bergerak di Rp 44–66 juta/klaim; rata-rata Rp 50.2 juta |
| **Lokasi RS** | Singapore: 22% volume → 49.9% total nilai (median 6x Indonesia) |
| **ICD Dominan** | Kanker (29.6% nilai) + Kardiovaskular (16.4% nilai) = 46% dari total |
| **Outlier** | 17 klaim >Rp 1 M (0.4% volume) = 9.5% total nilai |
| **Reporting Lag** | Rata-rata 66 hari; agregasi wajib pakai tanggal masuk RS |
| **Feature Importance** | Lokasi SG (42.26%), LOS (35.37%), Usia (10.78%) = 88.4% variasi severity |

> **→ Lanjut ke Bab 3.2 Machine Learning Training untuk pemodelan prediktif.**